# Neighbourhood Code Mapping Investigation

## Purpose

This notebook investigates whether `NEIGHBOURHOOD_CODE` from the City of Vancouver Property Tax Report can be translated into readable geography names using available project data and official documentation.

Notebook 08 confirmed that `NEIGHBOURHOOD_CODE` is the most suitable field for neighbourhood-level aggregation in this dataset, and Notebook 09 visualized the resulting 30-code output. However, the codes themselves are not human-readable. Before labelling any chart or output with neighbourhood names, an authoritative mapping source must be identified and validated.

This notebook does not produce any processed output files, visualizations, or final conclusions. It is an investigation-only step.

## Caveat

`NEIGHBOURHOOD_CODE` is a coded geography field. Do not treat codes as readable neighbourhood names unless an official mapping source is found.

## Imports

In [ ]:
import pandas as pd
from pathlib import Path

## Paths and Constants

All paths and configuration values are defined as named constants. `SAMPLE_SIZE` controls how many rows are loaded from the raw file to keep memory usage low during investigation. `CSV_SEPARATOR` is `";"` because the Property Tax Report uses semicolons as delimiters, confirmed in Notebook 01.

In [ ]:
RAW_PROPERTY_TAX_PATH      = Path("../data/raw/property_tax_report_raw.csv")
NEIGHBOURHOOD_OUTPUT_PATH  = Path("../data/processed/property_value_change_by_neighbourhood.csv")
CSV_SEPARATOR              = ";"
SAMPLE_SIZE                = 1000

print(f"RAW_PROPERTY_TAX_PATH     : {RAW_PROPERTY_TAX_PATH}")
print(f"NEIGHBOURHOOD_OUTPUT_PATH : {NEIGHBOURHOOD_OUTPUT_PATH}")
print(f"CSV_SEPARATOR             : {repr(CSV_SEPARATOR)}")
print(f"SAMPLE_SIZE               : {SAMPLE_SIZE:,}")

## Load Processed Neighbourhood-Code Output

We start from the already-processed output produced by Notebook 08. This gives us the full list of `neighbourhood_code` values used in the project, along with the property counts per code. Validation checks confirm the output is clean before we attempt any mapping investigation.

In [ ]:
neighbourhood_df = pd.read_csv(NEIGHBOURHOOD_OUTPUT_PATH)

print("Shape   :", neighbourhood_df.shape)
print("Columns :", neighbourhood_df.columns.tolist())
print()
print(f"Unique neighbourhood_code count  : {neighbourhood_df['neighbourhood_code'].nunique()}")
print(f"Duplicate neighbourhood_code count: {neighbourhood_df['neighbourhood_code'].duplicated().sum()}")
print(f"Missing neighbourhood_code count : {neighbourhood_df['neighbourhood_code'].isna().sum()}")
print(f"Total property_count             : {neighbourhood_df['property_count'].sum():,}")

## All Neighbourhood Codes

Display all `neighbourhood_code` values sorted ascending alongside their `property_count`. This gives a complete picture of the codes we need to map before treating any output as human-readable.

In [ ]:
codes_display = (
    neighbourhood_df[["neighbourhood_code", "property_count"]]
    .sort_values("neighbourhood_code", ascending=True)
    .reset_index(drop=True)
)

print(f"Total codes: {len(codes_display)}")
print()
print(codes_display.to_string(index=False))

## Inspect Raw Property Tax Columns

Load a 1,000-row sample from the raw Property Tax Report to inspect what other geography or name-related columns are available in the source data. We then run a keyword search over column names to identify any candidates that might carry readable neighbourhood or area labels.

In [ ]:
raw_sample = pd.read_csv(
    RAW_PROPERTY_TAX_PATH,
    sep=CSV_SEPARATOR,
    nrows=SAMPLE_SIZE,
)

print(f"Sample shape : {raw_sample.shape}")
print()
print("All columns:")
for col in raw_sample.columns:
    print(f"  {col}")

In [ ]:
GEO_KEYWORDS = [
    "neigh",
    "area",
    "local",
    "district",
    "zone",
    "geo",
    "address",
    "postal",
    "street",
    "code",
]

candidate_columns = [
    col for col in raw_sample.columns
    if any(kw in col.lower() for kw in GEO_KEYWORDS)
]

print(f"Candidate geography/name columns ({len(candidate_columns)} found):")
for col in candidate_columns:
    print(f"  {col}")

## Candidate Geography Field Summary

For each candidate column identified by the keyword search, calculate the non-null count, missing count, missing percentage, and number of unique values. Columns with lower missing rates and a moderate number of unique values are more likely to be useful as grouping or mapping fields.

In [ ]:
summary_rows = []

for col in candidate_columns:
    non_null   = int(raw_sample[col].notna().sum())
    missing    = int(raw_sample[col].isna().sum())
    unique     = int(raw_sample[col].nunique(dropna=True))
    missing_pct = round(missing / len(raw_sample) * 100, 2)
    summary_rows.append({
        "column"        : col,
        "non_null"      : non_null,
        "missing"       : missing,
        "missing_pct"   : missing_pct,
        "unique_values" : unique,
    })

candidate_summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values("missing_pct")
    .reset_index(drop=True)
)

print(candidate_summary_df.to_string(index=False))

## Search for Readable Labels in Raw Data

For each candidate column, display the top 20 most frequent values including missing. This reveals whether the field contains human-readable area names, numeric codes, or structured identifiers. No assumptions are made -- value previews only.

In [ ]:
TOP_N = 20

for col in candidate_columns:
    print(f"--- {col} ---")
    vc = raw_sample[col].value_counts(dropna=False).head(TOP_N)
    print(vc.to_string())
    print()

## Official Mapping Source Investigation

According to the City of Vancouver Open Data documentation for the Property Tax Report, `NEIGHBOURHOOD_CODE` is a 3-digit number assigned by BC Assessment. The City notes that BCA does not supply the City with the referencing neighbourhood name information, and users should contact BCA directly to obtain neighbourhood name information.

This confirms that a readable neighbourhood name mapping is not available within the Property Tax Report dataset itself. No other candidate column in the raw data supplies a readable neighbourhood label that maps directly to `NEIGHBOURHOOD_CODE`.

## Official Source Findings

Two official sources were consulted during this investigation. Both confirm that `NEIGHBOURHOOD_CODE` is a BCA-assigned coded field and that no readable neighbourhood name mapping is publicly available in the City of Vancouver open data.

### City of Vancouver Open Data — Property Tax Report documentation

`NEIGHBOURHOOD_CODE` is a 3-digit number assigned by BC Assessment (BCA) that identifies the neighbourhood for the folio. It is part of BC Assessment's assessment roll and geography structure. The City of Vancouver publishes the numeric code in the open data file but does not publish a corresponding readable neighbourhood name field. The City documentation explicitly states that BCA does not supply the City with the referencing neighbourhood name information, and that users should contact BCA directly to obtain neighbourhood name information.

### BC Assessment terminology

BC Assessment defines "Neighbourhood Code" as an identifier used by BCA to organize a jurisdiction into appraisal areas or like properties. This means the field is a BCA-coded appraisal and geography identifier, not a City of Vancouver neighbourhood designation. BC Assessment neighbourhoods are BCA-defined and may differ from municipal administrative areas, MLS zone boundaries, or other geography sources. The field should be treated as a BCA-coded appraisal geography field, not automatically as a readable or publicly labelled City of Vancouver neighbourhood name.

### Summary of findings

- `NEIGHBOURHOOD_CODE` is an official coded field from BC Assessment, not a City of Vancouver neighbourhood label.
- It is part of BC Assessment's assessment roll/geography structure.
- The City of Vancouver Property Tax Report publishes the numeric code but does not publish a corresponding readable neighbourhood name field.
- BCA does not supply the City with the referencing neighbourhood name information.
- BC Assessment neighbourhoods are BCA-defined and may differ from municipal areas, MLS, or other sources.
- A readable neighbourhood description likely exists inside BC Assessment restricted or commercial data products (such as Data Advice or paid property information products), but no free public downloadable lookup was found during this investigation.
- The City of Vancouver's 22 Local Areas are a different geography from the 30 BCA neighbourhood codes and must not be treated as a substitute or direct mapping.

## Mapping Status

**No official public mapping found.**

A code-to-name mapping for `NEIGHBOURHOOD_CODE` likely exists in BC Assessment restricted or commercial data products — such as BCA's Data Advice service or paid property information products — but it is not available in the public City of Vancouver open-data files used for this project. No free, downloadable, official lookup table was found during the investigation conducted in this notebook.

## Methodological Implications

The absence of a public mapping table has the following consequences for this project.

- **The project can analyze patterns by `neighbourhood_code`.** Aggregated metrics such as median assessed value change, increase/decrease shares, and property counts are valid at the neighbourhood-code level and are based entirely on public data.
- **The project cannot honestly label the codes with readable names from public sources alone.** Labelling any of the 30 codes as Kitsilano, Downtown, Mount Pleasant, or any other neighbourhood name would require an official lookup that does not exist in the publicly available data used for this project.
- **City of Vancouver Local Areas are not equivalent to the 30 BCA codes.** The City's 22 Local Areas are a different geography system. They cannot be used as a direct substitute for BCA neighbourhood codes without a verified official crosswalk.
- **The current chart and processed output should be described as BCA-coded geography or neighbourhood-code-level analysis**, not as a named neighbourhood analysis. Titles, captions, and axis labels should reference `neighbourhood_code` or "coded geography area," not named neighbourhoods.
- **A separate City Local Area analysis could be explored later** using spatial assignment or an official crosswalk, but that would be a distinct geography level and a separate analytical step. It should not be treated as equivalent to the 30 BCA neighbourhood codes.

## Recommended Project Decision

**Decision: continue using `neighbourhood_code` as a coded BCA geography field. Do not create a manual mapping table at this stage.**

No official `NEIGHBOURHOOD_CODE` to readable neighbourhood name lookup has been identified in the project data or City of Vancouver Property Tax Report documentation. Creating a manual or inferred mapping would introduce unverifiable geographic labelling that cannot be reproducibly sourced from the public data used in this project.

This keeps the project reproducible from public data sources and avoids unverifiable geographic labelling. All outputs, charts, and documentation produced from this dataset will refer to this geography level as `neighbourhood_code` or "BCA-coded neighbourhood/appraisal geography area."

## Suggested Documentation Wording

The following wording is safe to use in README, methodology notes, or any portfolio-facing documentation for this project. It accurately reflects the findings of this investigation without guessing or implying readable neighbourhood names.

> "The Property Tax Report identifies geography using BC Assessment's numeric `NEIGHBOURHOOD_CODE`. No free official public lookup mapping these codes to readable neighbourhood names was found during research. Therefore, this project reports these results by anonymous BCA neighbourhood code rather than by neighbourhood name. These codes should not be treated as equivalent to the City of Vancouver's 22 Local Areas."

## Preliminary Mapping Conclusion

### Investigation findings

The raw Property Tax sample (1,000 rows, 30 columns) was inspected for candidate geography and name fields. Five columns matched the keyword search: `ZONING_DISTRICT`, `DISTRICT_LOT`, `STREET_NAME`, `PROPERTY_POSTAL_CODE`, and `NEIGHBOURHOOD_CODE`.

| Column | Non-null | Missing | Missing % | Unique values | Assessment |
|---|---|---|---|---|---|
| `ZONING_DISTRICT` | 1,000 | 0 | 0.00 % | 158 | Zoning classification — not neighbourhood geography |
| `STREET_NAME` | 1,000 | 0 | 0.00 % | 274 | Too granular for neighbourhood-level labelling |
| `NEIGHBOURHOOD_CODE` | 1,000 | 0 | 0.00 % | 30 | Only suitable grouped geography field; codes are not readable names |
| `PROPERTY_POSTAL_CODE` | 984 | 16 | 1.60 % | 724 | Too granular; forward sortation area alone would add derived complexity |
| `DISTRICT_LOT` | 977 | 23 | 2.30 % | 66 | Legal land description unit; not interpretable as a neighbourhood label |

**`NEIGHBOURHOOD_CODE` is the only suitable grouped geography field available in the Property Tax sample.** The raw data does not include a readable neighbourhood name field that maps directly to it.

- `ZONING_DISTRICT` represents zoning classification, not neighbourhood geography. It cannot be used as a neighbourhood name substitute.
- `STREET_NAME` and `PROPERTY_POSTAL_CODE` are address-level fields. They are too granular for neighbourhood-level labelling and would produce hundreds of narrow, difficult-to-interpret groups.
- `DISTRICT_LOT` is a legal land description unit from the Torrens title system. It is less interpretable for a business audience and does not provide readable neighbourhood names.

### Official source note

According to the City of Vancouver Open Data documentation for the Property Tax Report, `NEIGHBOURHOOD_CODE` is a 3-digit number assigned by BC Assessment. The City notes that BCA does not supply the City with the referencing neighbourhood name information, and users should contact BCA to obtain neighbourhood name information. No readable neighbourhood name field is provided within the dataset.

### Boundary mismatch caution

The City of Vancouver Local Area Boundary dataset contains 22 local areas, while this Property Tax Report output contains 30 neighbourhood codes. Local areas and neighbourhood codes should not be assumed to be equivalent without an official crosswalk.

### Decision

The following five conclusions apply to this project and can be audited against the evidence above:

1. **No official public mapping found.** No free, official, public lookup table mapping `NEIGHBOURHOOD_CODE` to readable neighbourhood names was identified in the project data or City of Vancouver Property Tax Report documentation.
2. **A code-to-name mapping may exist in BC Assessment restricted or commercial data products**, but it is not available in the public City of Vancouver open-data files used for this project.
3. **The City of Vancouver's 22 Local Areas are a different geography from the 30 BCA neighbourhood codes** and should not be treated as equivalent without a verified official crosswalk.
4. **Do not create a manual mapping from street names, postal codes, or addresses.** `STREET_NAME`, `PROPERTY_POSTAL_CODE`, and `DISTRICT_LOT` do not carry readable neighbourhood name information and cannot be used to infer or reverse-engineer a code-to-name lookup.
5. **Continue using `neighbourhood_code` as a BCA-coded geography field.** All project outputs, charts, and documentation will refer to this geography level as `neighbourhood_code` or "BCA-coded neighbourhood/appraisal geography area" until an authoritative mapping source is obtained.